## Environment Setup & Imports
Configure project paths and import core libraries alongside the modular `src/` engines.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_raw_data
from src.cleaning import clean_data, print_cleaning_report
from src.eda import generate_eda_report, print_eda_report
from src.feature_engineering import engineer_features
from src.preprocessing import split_data, get_feature_types, build_preprocessor

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)

## Data Ingestion & Hygiene Check
Load raw data and run sanitization: resolve whitespace strings in `TotalCharges`, encode the target, and drop non-predictive identifiers.

In [2]:
# 1. Ingest raw dataset (downloads automatically if not cached locally)
df_raw = load_raw_data(data_dir=PROJECT_ROOT / "data" / "raw")

# 2. Sanitize and view hygiene report
df_clean, clean_report = clean_data(df_raw)
print_cleaning_report(clean_report)

[DataLoader] Using cached raw dataset from: D:\AI-ML\NOTES\logistic-regression\telco-churn-guardian\data\raw\Telco-Customer-Churn.csv
[DataLoader] Ingested raw shape: 7043 rows, 21 columns.
               DATA SANITIZATION REPORT                     
  • Rows Retained                 : 7043
  • Columns Retained              : 20
  • TotalCharges Whitespace Fixed : 11
  • TotalCharges Imputed (with 0) : 11
  • Identifier Dropped            : customerID
  • Target Variable Encoded       : {'Yes': 1, 'No': 0}


## Automated EDA Profile
Audit target class distribution, confirm zero remaining missing values, and check baseline linear correlations with churn.

In [3]:
df_clean.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [4]:
eda_report = generate_eda_report(df_clean, target_col="Churn")
print_eda_report(eda_report)

                 AUTOMATED DATA PROFILE REPORT                   
Dataset Dimensions: 7043 rows × 19 input features
-----------------------------------------------------------------
TARGET CLASS DISTRIBUTION ('Churn'):
  • Retained (0) : 5,174 (73.46%)
  • Churned  (1) : 1,869 (26.54%)
  • Imbalance Ratio: 1 positive : 2.77 negatives
-----------------------------------------------------------------
DATA INTEGRITY & CARDINALITY:
  • Missing Values Detected : Zero missing
  • Numerical Features (4) : ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
  • Categorical Features (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
-----------------------------------------------------------------
LINEAR CORRELATION WITH CHURN (Numeric Features):
  • MonthlyCharges        : +0.1934 (▲ Risk fac

## Feature Engineering
Generate domain-informed interaction features (`Charge_Discrepancy`, `Total_Services_Subscribed`, `Tenure_Cohort`) to assist the linear decision boundary.

In [6]:
df_featured, new_cols = engineer_features(df_clean)
print(f"Engineered features added: {new_cols}")

# Preview new features alongside core billing attributes
df_featured[['tenure', 'MonthlyCharges', 'TotalCharges'] + new_cols].head()

Engineered features added: ['Charge_Discrepancy', 'Total_Services_Subscribed', 'Tenure_Cohort']


,tenure,MonthlyCharges,TotalCharges,Charge_Discrepancy,Total_Services_Subscribed,Tenure_Cohort
0,1,29.85,29.85,0.000000,1,0-12m
1,34,56.95,1889.50,1.376471,3,24-48m
2,2,53.85,108.15,-0.225000,3,0-12m
3,45,42.30,1840.75,1.394444,3,24-48m
4,2,70.70,151.65,-5.125000,1,0-12m


## Data Splitting & Preprocessing Pipeline Setup
Perform a stratified 80/20 train/test split to preserve the ~26.6% churn ratio in both sets, and build the leakage-free `ColumnTransformer`.

In [7]:
# 1. Stratified train/test split
X_train, X_test, y_train, y_test = split_data(
    df_featured, 
    target_col="Churn", 
    test_size=0.20, 
    random_state=42
)

# 2. Extract column types
numeric_features, categorical_features = get_feature_types(X_train)

# 3. Build ColumnTransformer (StandardScaler for numerics, OneHotEncoder for categoricals)
preprocessor = build_preprocessor(numeric_features, categorical_features)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.4f} | Test churn rate: {y_test.mean():.4f}")

X_train shape: (5634, 22) | X_test shape: (1409, 22)
Train churn rate: 0.2654 | Test churn rate: 0.2654


## Fit & Inspect Preprocessor Transformation
Fit the `ColumnTransformer` on `X_train` and transform both train and test sets to verify that scaling and one-hot encoding execute cleanly into a pure numeric matrix.

In [10]:
# 1. Fit on X_train ONLY, then transform X_train and X_test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 2. Extract the generated feature names (after one-hot encoding)
feature_names = preprocessor.get_feature_names_out()

print(f"Original features count    : {X_train.shape[1]}")
print(f"Processed features count   : {X_train_processed.shape[1]}")
print(f"X_train_processed shape    : {X_train_processed.shape}")
print(f"X_test_processed shape     : {X_test_processed.shape}")
print(f"All values are float64     : {X_train_processed.dtype}")

# 3. Preview first 5 processed rows as a DataFrame to verify the numbers
pd.DataFrame(X_train_processed, columns=feature_names).head()

Original features count    : 22
Processed features count   : 35
X_train_processed shape    : (5634, 35)
X_test_processed shape     : (1409, 35)
All values are float64     : float64


,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,num__Charge_Discrepancy,num__Total_Services_Subscribed,cat__gender_Male,cat__Partner_Yes,cat__Dependents_Yes,cat__PhoneService_Yes,...,cat__StreamingMovies_Yes,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_Yes,cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,cat__Tenure_Cohort_12-24m,cat__Tenure_Cohort_24-48m,cat__Tenure_Cohort_48m+
0,-0.441773,0.102371,-0.521976,-0.262257,0.169324,-0.184954,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,-0.441773,-0.711743,0.337478,-0.503635,-0.549707,-0.667823,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-0.441773,-0.793155,-0.809013,-0.749883,-1.568978,-0.184954,1.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,-0.441773,-0.263980,0.284384,-0.172722,0.048737,0.780784,0.0,1.0,0.0,1.0,...,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
4,-0.441773,-1.281624,-0.676279,-0.989374,-0.016365,-1.150692,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


## Naive Baseline Model
Fit a `DummyClassifier` that always predicts the majority class ("No Churn") to establish the minimum performance floor our real model must beat.

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# Baseline strategy: always predict the most frequent class (0: Retained)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_processed, y_train)

y_pred_dummy = dummy.predict(X_test_processed)
y_proba_dummy = dummy.predict_proba(X_test_processed)[:, 1]

print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_dummy):.4f}")
print(f"Baseline Recall  : {recall_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print(f"Baseline ROC-AUC : {roc_auc_score(y_test, y_proba_dummy):.4f}")

Baseline Accuracy: 0.7346
Baseline Recall  : 0.0000
Baseline ROC-AUC : 0.5000
